# 1. Imports and Environment Setup

In [11]:
import os
import re
import json
import time
import textwrap
from pathlib import Path

import requests
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import InferenceClient # Hugging Face Inference API client
from openai import OpenAI # OpenAI API client

# Load environment variables from a local .env file if present
load_dotenv()

# Project paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

# Create common folders if they do not already exist
DATA_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

# Environment variables
HF_TOKEN = os.getenv("HF_TOKEN", "")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "")
LLAMA_MODEL = os.getenv("LLAMA_MODEL", "")


# Basic environment checks
print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Outputs directory:", OUTPUTS_DIR)


if HF_TOKEN:
    print("HF_TOKEN loaded.")
else:
    print("HF_TOKEN not found in environment.")

if OPENAI_API_KEY:
    print("OPENAI_API_KEY loaded.")
else:
    print("OPENAI_API_KEY not found in environment.")

if OPENAI_MODEL:
    print("OPENAI_MODEL:", OPENAI_MODEL)
else:    print("OPENAI_MODEL not set.")

if LLAMA_MODEL:
    print("LLAMA_MODEL:", LLAMA_MODEL)
else:
    print("LLAMA_MODEL not set.")

Project root: c:\Users\Yuna\PolishCV
Data directory: c:\Users\Yuna\PolishCV\data
Outputs directory: c:\Users\Yuna\PolishCV\outputs
HF_TOKEN loaded.
OPENAI_API_KEY loaded.
OPENAI_MODEL: gpt-5.4
LLAMA_MODEL: meta-llama/Llama-3.2-1B-Instruct


# 2.  AI Configuration

In [12]:
PROVIDER = "huggingface"   # current plan: use Hugging Face for both models

MODEL_OPTIONS = {
    "openai": OPENAI_MODEL if OPENAI_MODEL else "gpt-5.4",
    "llama": LLAMA_MODEL if LLAMA_MODEL else "meta-llama/Llama-3.2-3B-Instruct"
}

# Generation settings
GENERATION_CONFIG = {
    "max_new_tokens": 100,
    "temperature": 0.3,
    "top_p": 0.9,
    "do_sample": True,
    "return_full_text": False
}


# Input limits
MIN_RESUME_CHARS = 100
MIN_JOB_DESCRIPTION_CHARS = 100
MAX_INPUT_CHARS = 12000


# File paths
TEST_CASES_PATH = DATA_DIR / "test_cases.json"
VERIFICATION_CASES_PATH = DATA_DIR / "verification_cases.json"
SAMPLE_INPUTS_PATH = DATA_DIR / "sample_inputs.json"
EVAL_RESULTS_PATH = OUTPUTS_DIR / "eval_results.csv"
SAMPLE_OUTPUTS_PATH = OUTPUTS_DIR / "sample_model_outputs.json"


# Evaluation settings
SUPPORTED_TASKS = [
    "resume_feedback",
    "resume_tailor"
]

DEFAULT_TASK = "resume_feedback"

RUBRIC_DIMENSIONS = [
    "ats_alignment",
    "factual_faithfulness",
    "relevance_to_job_description",
    "clarity_professionalism",
    "usefulness_of_feedback"
]


# Safety / trustworthiness settings
ANTI_HALLUCINATION_RULES = {
    "no_new_jobs": True,
    "no_new_dates": True,
    "no_new_metrics": True,
    "no_new_certifications": True,
    "preserve_user_facts": True
}

print("Provider:", PROVIDER)
print("Available models:", MODEL_OPTIONS)
print("Supported tasks:", SUPPORTED_TASKS)
print("Evaluation results path:", EVAL_RESULTS_PATH)

Provider: huggingface
Available models: {'openai': 'gpt-5.4', 'llama': 'meta-llama/Llama-3.2-1B-Instruct'}
Supported tasks: ['resume_feedback', 'resume_tailor']
Evaluation results path: c:\Users\Yuna\PolishCV\outputs\eval_results.csv


# 3. Utility Functions

In [13]:
#Utility helper functions for the PolishCV project

from typing import Any, Dict, List, Optional


def clean_text(text: str) -> str:
    """
    Clean and normalize input text.

    What it should do:
    - handle None or empty input safely
    - strip leading/trailing whitespace
    - normalize repeated spaces
    - normalize repeated blank lines
    """
    pass


def validate_inputs(resume_text: str, job_description: str) -> Dict[str, Any]:
    """
    Validate resume and job description inputs.

    Returns a dictionary such as:
    {
        "is_valid": True/False,
        "errors": [ ... ]
    }

    Checks to include:
    - resume is not empty
    - job description is not empty
    - both meet minimum character count
    - both stay under max input limit
    """
    pass


def truncate_text(text: str, max_chars: int = MAX_INPUT_CHARS) -> str:
    """
    Truncate text if it exceeds the allowed maximum length.
    """
    pass


def extract_keywords_from_jd(job_description: str, top_n: int = 20) -> List[str]:
    """
    Extract simple keywords from a job description.

    This can be a lightweight keyword extractor for:
    - skills
    - tools
    - technologies
    - repeated important terms

    Keep this simple for now.
    """
    pass


def safe_json_loads(text: str) -> Optional[Dict[str, Any]]:
    """
    Safely parse JSON returned by a model.

    Returns:
    - parsed dictionary if successful
    - None if parsing fails
    """
    pass


def parse_model_output(raw_output: str) -> Dict[str, Any]:
    """
    Convert raw model output into a structured dictionary.

    Expected target structure could look like:
    {
        "summary": "",
        "rewritten_experience": [],
        "missing_keywords": [],
        "feedback": [],
        "gap_suggestions": [],
        "risk_flags": []
    }

    For now, this is a fallback parser skeleton.
    """
    pass


def format_bullet_list(items: List[str]) -> str:
    """
    Format a list of strings as bullet points for display.
    """
    pass


def compute_latency(start_time: float, end_time: float) -> float:
    """
    Compute elapsed time in seconds for a model request.
    """
    pass


def load_json_file(file_path: Path) -> Any:
    """
    Load JSON data from a file path.
    """
    pass


def save_json_file(data: Any, file_path: Path) -> None:
    """
    Save JSON data to a file path.
    """
    pass


def append_results_to_csv(row: Dict[str, Any], file_path: Path) -> None:
    """
    Append one evaluation result row to a CSV file.

    Useful for logging:
    - case_id
    - model_name
    - task
    - before_score
    - after_score
    - score_delta
    - latency
    - notes
    """
    pass

#  4. System Prompts

In [14]:
# Purpose: Prompt templates for the PolishCV project

def build_system_instruction() -> str:
    """
    Shared system-style instruction used across prompt types.
    """
    return """
You are an experienced technical recruiter and resume reviewer focused on entry-level software engineering roles.

Your job is to improve resumes in a helpful, professional, and trustworthy way.

Important rules:
1. Do not invent facts.
2. Do not add new companies, job titles, dates, certifications, degrees, or metrics unless they are explicitly provided by the user.
3. Do not exaggerate the candidate's experience.
4. Tailor suggestions to the provided job description only.
5. Prefer clear, concise, ATS-friendly language.
6. Explain your suggestions in a way that a human user can review and verify.
7. If important information is missing, say so instead of making assumptions.
""".strip()


def build_output_format_instruction() -> str:
    """
    Instruction telling the model to return structured JSON.
    """
    return """
Return your answer as valid JSON with the following keys:
{
  "summary": "short overview of your assessment",
  "rewritten_experience": ["list of improved resume bullet points or revised lines"],
  "missing_keywords": ["list of important keywords or skills missing from the resume"],
  "feedback": ["list of concrete suggestions for improvement"],
  "gap_suggestions": ["list of suggestions for addressing weak areas or experience gaps"],
  "risk_flags": ["list of possible trustworthiness or accuracy concerns the user should review"]
}

Rules for formatting:
- Return JSON only.
- Do not include markdown.
- Do not include code fences.
- If a field has no content, return an empty list or empty string.
""".strip()


def build_resume_feedback_prompt(resume_text: str, job_description: str) -> str:
    """
    Build a prompt for reviewing the user's current resume
    against a target job description.
    """
    system_instruction = build_system_instruction()
    output_instruction = build_output_format_instruction()

    return f"""
{system_instruction}

Task:
Review the user's resume for an entry-level software engineering job.
Compare it against the target job description and provide ATS-friendly feedback.

Resume:
{resume_text}

Target Job Description:
{job_description}

What to do:
- identify strengths
- identify missing or weak keywords and skills
- point out unclear, weak, or overly generic wording
- suggest improvements that better align the resume with the job description
- suggest ways to strengthen weak areas without inventing experience
- flag anything that might be inaccurate, misleading, or too vague

{output_instruction}
""".strip()


def build_resume_tailor_prompt(resume_text: str, job_description: str) -> str:
    """
    Build a prompt for tailoring the resume to a job description.
    """
    system_instruction = build_system_instruction()
    output_instruction = build_output_format_instruction()

    return f"""
{system_instruction}

Task:
Tailor the user's resume for the target entry-level software engineering role.

Resume:
{resume_text}

Target Job Description:
{job_description}

What to do:
- rewrite parts of the resume to better align with the job description
- improve clarity, relevance, and ATS-friendly wording
- preserve the user's original facts
- do not invent experience or metrics
- highlight important missing keywords
- provide feedback explaining the most important changes

{output_instruction}
""".strip()


def build_gap_suggestions_prompt(resume_text: str, job_description: str) -> str:
    """
    Optional lightweight prompt for suggestions on how a user
    could strengthen weak areas without fabricating experience.
    """
    system_instruction = build_system_instruction()
    output_instruction = build_output_format_instruction()

    return f"""
{system_instruction}

Task:
Suggest realistic ways the user could strengthen weak areas in the resume
for the target entry-level software engineering role.

Resume:
{resume_text}

Target Job Description:
{job_description}

What to do:
- identify possible experience or skill gaps
- suggest realistic next steps such as projects, internships, volunteer work,
  open-source work, research, coursework, or certifications
- do not claim the user already has these experiences
- do not rewrite the entire resume
- keep suggestions practical and appropriate for a student or recent graduate

{output_instruction}
""".strip()


def build_prompt(task_name: str, resume_text: str, job_description: str) -> str:
    """
    Dispatch function for selecting the correct prompt template.
    """
    if task_name == "resume_feedback":
        return build_resume_feedback_prompt(resume_text, job_description)
    elif task_name == "resume_tailor":
        return build_resume_tailor_prompt(resume_text, job_description)
    elif task_name == "gap_suggestions":
        return build_gap_suggestions_prompt(resume_text, job_description)
    else:
        raise ValueError(f"Unsupported task_name: {task_name}")

# 5. Model Inference Functions

In [15]:
# Model inference functions for the PolishCV project

from typing import Tuple

def get_hf_client() -> InferenceClient:
    """
    Create a Hugging Face inference client using the token from .env
    """
    if not HF_TOKEN:
        raise ValueError("HF_TOKEN is missing. Please add it to your .env file.")
    return InferenceClient(api_key=HF_TOKEN)



def call_hf_model(model_id: str, prompt: str) -> Dict[str, Any]:
    """
    Call a Hugging Face model using InferenceClient and return a structured response.
    """
    client = get_hf_client()
    start_time = time.time()

    try:
        completion = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "user", "content": prompt}
            ],
            max_tokens=GENERATION_CONFIG.get("max_new_tokens", 500),
            temperature=GENERATION_CONFIG.get("temperature", 0.3),
            top_p=GENERATION_CONFIG.get("top_p", 0.9),
        )
        end_time = time.time()

        generated_text = ""
        if completion and getattr(completion, "choices", None):
            generated_text = completion.choices[0].message.content or ""

        if not generated_text.strip():
            return {
                "success": False,
                "model_id": model_id,
                "raw_text": "",
                "latency_seconds": compute_latency(start_time, end_time),
                "error": "Model response was empty."
            }

        return {
            "success": True,
            "model_id": model_id,
            "raw_text": generated_text.strip(),
            "latency_seconds": compute_latency(start_time, end_time),
            "error": None
        }

    except Exception as exc:
        end_time = time.time()
        return {
            "success": False,
            "model_id": model_id,
            "raw_text": "",
            "latency_seconds": compute_latency(start_time, end_time),
            "error": str(exc)
        }

def get_openai_client() -> OpenAI:
    """
    Create an OpenAI client using the API key from .env
    """
    if not OPENAI_API_KEY:
        raise ValueError("OPENAI_API_KEY is missing. Please add it to your .env file.")
    return OpenAI(api_key=OPENAI_API_KEY)


def call_openai_model(model_id: str, prompt: str) -> Dict[str, Any]:
    """
    Call an OpenAI model and return a structured response.
    """
    client = get_openai_client()
    start_time = time.time()

    try:
        completion = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "user", "content": prompt}
            ],
            max_completion_tokens=GENERATION_CONFIG.get("max_new_tokens", 500),
            temperature=GENERATION_CONFIG.get("temperature", 0.3),
            top_p=GENERATION_CONFIG.get("top_p", 0.9),
        )
        end_time = time.time()

        generated_text = ""
        if completion and getattr(completion, "choices", None):
            generated_text = completion.choices[0].message.content or ""

        if not generated_text.strip():
            return {
                "success": False,
                "model_id": model_id,
                "raw_text": "",
                "latency_seconds": compute_latency(start_time, end_time),
                "error": "Model response was empty."
            }

        return {
            "success": True,
            "model_id": model_id,
            "raw_text": generated_text.strip(),
            "latency_seconds": compute_latency(start_time, end_time),
            "error": None
        }

    except Exception as exc:
        end_time = time.time()
        return {
            "success": False,
            "model_id": model_id,
            "raw_text": "",
            "latency_seconds": compute_latency(start_time, end_time),
            "error": str(exc)
        }


def run_openai(prompt: str) -> Dict[str, Any]:
    model_id = MODEL_OPTIONS["openai"]
    return call_openai_model(model_id=model_id, prompt=prompt)

def run_llama(prompt: str) -> Dict[str, Any]:
    """
    Run the configured Llama model.
    """
    model_id = MODEL_OPTIONS["llama"]
    return call_hf_model(model_id=model_id, prompt=prompt)


def run_selected_model(model_name: str, prompt: str) -> Dict[str, Any]:
    """
    Dispatch inference based on the selected model name.
    """
    model_name = model_name.lower().strip()

    if model_name == "openai":
        return run_openai(prompt)
    elif model_name == "llama":
        return run_llama(prompt)
    else:
        raise ValueError(f"Unsupported model_name: {model_name}")


def generate_for_task(model_name: str, task_name: str, resume_text: str, job_description: str) -> Dict[str, Any]:
    """
    End-to-end helper function:
    - builds the task prompt
    - calls the selected model
    - returns the raw model response
    """
    prompt = build_prompt(
        task_name=task_name,
        resume_text=resume_text,
        job_description=job_description
    )

    result = run_selected_model(model_name=model_name, prompt=prompt)
    result["task_name"] = task_name
    result["prompt"] = prompt
    return result

# 6. ATS Alignment Scoring

In [16]:
# Purpose: ATS-alignment scoring functions for the PolishCV project


STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "in",
    "is", "it", "of", "on", "or", "that", "the", "to", "with", "will",
    "this", "your", "you", "our", "their", "they", "we", "was", "were",
    "has", "have", "had", "but", "not", "if", "into", "than", "then",
    "so", "such", "using", "use", "used", "about", "can", "should"
}

COMMON_TECH_KEYWORDS = {
    "python", "java", "javascript", "typescript", "c++", "c", "sql", "html",
    "css", "react", "node", "node.js", "flask", "django", "git", "github",
    "aws", "docker", "kubernetes", "linux", "api", "rest", "tensorflow",
    "pytorch", "pandas", "numpy", "machine learning", "data structures",
    "algorithms", "oop", "debugging", "testing", "streamlit"
}

ACTION_VERBS = {
    "built", "developed", "designed", "implemented", "created", "optimized",
    "improved", "deployed", "tested", "analyzed", "collaborated", "led",
    "automated", "engineered", "debugged", "refactored", "integrated"
}

COMMON_RESUME_SECTIONS = {
    "education", "experience", "projects", "skills", "summary"
}


def normalize_text(text: str) -> str:
    """
    Lowercase and lightly normalize text for matching.
    """
    if not text:
        return ""
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def tokenize_text(text: str) -> List[str]:
    """
    Tokenize text into simple lowercase words.
    """
    text = normalize_text(text)
    tokens = re.findall(r"[a-zA-Z0-9\+\.\#\-]+", text)
    return tokens


def get_keyword_set(text: str) -> set:
    """
    Convert text into a basic keyword set after stopword filtering.
    """
    tokens = tokenize_text(text)
    keywords = {
        token for token in tokens
        if token not in STOPWORDS and len(token) > 1
    }
    return keywords


def extract_important_jd_keywords(job_description: str, top_n: int = 20) -> List[str]:
    """
    Extract a simple list of important keywords from the job description.

    Strategy:
    - tokenize
    - remove stopwords
    - count frequency
    - prioritize tech-related keywords when present
    """
    tokens = tokenize_text(job_description)
    filtered = [t for t in tokens if t not in STOPWORDS and len(t) > 1]

    counts: Dict[str, int] = {}
    for token in filtered:
        counts[token] = counts.get(token, 0) + 1

    sorted_tokens = sorted(
        counts.items(),
        key=lambda item: (
            item[0] not in COMMON_TECH_KEYWORDS,   # prefer known tech terms first
            -item[1],                              # then by frequency
            item[0]
        )
    )

    return [token for token, _ in sorted_tokens[:top_n]]


def keyword_overlap_score(resume_text: str, job_description: str, top_n: int = 20) -> float:
    """
    Score overlap between important JD keywords and resume keywords.
    Returns a value from 0 to 100.
    """
    jd_keywords = set(extract_important_jd_keywords(job_description, top_n=top_n))
    resume_keywords = get_keyword_set(resume_text)

    if not jd_keywords:
        return 0.0

    overlap = jd_keywords.intersection(resume_keywords)
    score = (len(overlap) / len(jd_keywords)) * 100
    return round(score, 2)


def tech_keyword_score(resume_text: str, job_description: str) -> float:
    """
    Measure coverage of common technical keywords that appear in the JD.
    Returns a value from 0 to 100.
    """
    jd_text = normalize_text(job_description)
    resume_text_norm = normalize_text(resume_text)

    jd_tech_terms = [kw for kw in COMMON_TECH_KEYWORDS if kw in jd_text]

    if not jd_tech_terms:
        return 100.0

    matched = [kw for kw in jd_tech_terms if kw in resume_text_norm]
    score = (len(matched) / len(jd_tech_terms)) * 100
    return round(score, 2)


def action_verb_score(resume_text: str) -> float:
    """
    Measure how much the resume uses action-oriented language.
    Returns a value from 0 to 100.
    """
    resume_tokens = set(tokenize_text(resume_text))
    matched = ACTION_VERBS.intersection(resume_tokens)

    if not ACTION_VERBS:
        return 0.0

    score = min((len(matched) / 8) * 100, 100)  # cap after a reasonable number
    return round(score, 2)


def section_coverage_score(resume_text: str) -> float:
    """
    Check whether common resume sections are present.
    Returns a value from 0 to 100.
    """
    resume_text_norm = normalize_text(resume_text)
    present_sections = [section for section in COMMON_RESUME_SECTIONS if section in resume_text_norm]

    score = (len(present_sections) / len(COMMON_RESUME_SECTIONS)) * 100
    return round(score, 2)


def compute_ats_score(resume_text: str, job_description: str) -> Dict[str, Any]:
    """
    Compute a simple ATS-alignment score with component breakdown.

    Weighted components:
    - keyword overlap: 45%
    - technical keyword coverage: 30%
    - action verbs: 15%
    - section coverage: 10%
    """
    overlap = keyword_overlap_score(resume_text, job_description)
    tech = tech_keyword_score(resume_text, job_description)
    action = action_verb_score(resume_text)
    sections = section_coverage_score(resume_text)

    overall = (
        0.45 * overlap +
        0.30 * tech +
        0.15 * action +
        0.10 * sections
    )

    return {
        "overall_score": round(overall, 2),
        "keyword_overlap_score": overlap,
        "tech_keyword_score": tech,
        "action_verb_score": action,
        "section_coverage_score": sections,
        "important_jd_keywords": extract_important_jd_keywords(job_description, top_n=20)
    }


def compare_scores(original_resume: str, revised_resume: str, job_description: str) -> Dict[str, Any]:
    """
    Compare ATS-alignment scores before and after revision.
    """
    before = compute_ats_score(original_resume, job_description)
    after = compute_ats_score(revised_resume, job_description)

    delta = round(after["overall_score"] - before["overall_score"], 2)

    return {
        "before": before,
        "after": after,
        "score_delta": delta
    }


def score_band(score: float) -> str:
    """
    Map a numeric score to the project's interpretation bands.
    """
    if score >= 80:
        return "Excellent / Strong match"
    elif score >= 60:
        return "Good / Moderate match"
    return "Weak match"

In [17]:
import re

# Simple scoring smoke test
sample_resume = """
MAYA PATEL
San Jose, CA | maya.patel@email.com | linkedin.com/in/mayapatel | github.com/mayapatel

SUMMARY
Motivated Software Engineer and recent Computer Science graduate with experience building scalable web applications, AI-powered features, and internal productivity tools through internships, research, and academic projects. Proficient in JavaScript, React, Node.js, Python, and Java, with hands-on experience in GraphQL, Spring Boot, NoSQL databases, and cloud deployment. Strong collaborator with a track record of shipping high-quality software in fast-paced environments.

SKILLS
Languages: JavaScript, TypeScript, Python, Java, SQL
Frameworks: React, Node.js, Express, Spring Boot, GraphQL, Next.js
Tools: Git, Docker, AWS, GCP, MongoDB, PostgreSQL, Redis, Jest, Cypress
Concepts: REST APIs, GraphQL APIs, NoSQL, data structures, algorithms, CI/CD, GenAI, LLM integration, scalable application design

EXPERIENCE

Software Engineering Intern | CloudBridge Systems, San Francisco, CA | Jun 2025 – Aug 2025
- Built employee-facing dashboard features in React and Node.js used by internal operations teams to manage workflow requests
- Developed REST and GraphQL API endpoints for task status, approvals, and reporting, reducing manual processing time by 25%
- Integrated OpenAI-powered smart text suggestions into internal forms, improving form completion speed and consistency
- Partnered with product managers and designers to iterate on UX flows, accessibility, and performance improvements
- Wrote unit and integration tests and contributed to CI/CD workflows using GitHub Actions

Undergraduate Research Assistant, Applied AI Lab | University of California, Davis | Sep 2024 – May 2025
- Prototyped LLM-powered semantic search and question-answering workflows for enterprise-style document retrieval
- Built Python services to preprocess structured and unstructured data for retrieval pipelines using vector embeddings
- Evaluated prompt quality, latency, and accuracy across multiple models and presented findings to faculty and student teams
- Collaborated with 3 researchers to design experiments and improve reliability of AI-generated responses

Teaching Assistant, Data Structures | University of California, Davis | Jan 2024 – May 2024
- Guided 60+ students on algorithms, recursion, graph traversal, and complexity analysis
- Led weekly debugging sessions and code reviews focused on clean code, testing, and problem solving

EDUCATION
B.S. in Computer Science | University of California, Davis | 2026
GPA: 3.8/4.0 | Dean’s List | Relevant Coursework: Data Structures, Algorithms, Database Systems, Software Engineering, Machine Learning, Distributed Systems

CERTIFICATIONS
- AWS Certified Cloud Practitioner (2025)
- GraphQL with React: Professional Certificate (2025)

PROJECTS

AI Workflow Assistant | github.com/mayapatel/ai-workflow-assistant
- Built an enterprise-style internal productivity tool with React, Node.js, GraphQL, and MongoDB
- Designed chat, autocomplete, and smart suggestion UX flows powered by LLM APIs
- Implemented role-based access, workflow history, and prompt logging for reliability and auditing
- Deployed cloud-hosted services and supported 1,000+ test interactions during demo evaluation

Knowledge Hub API Platform | github.com/mayapatel/knowledge-hub
- Developed Spring Boot microservices with GraphQL and REST endpoints for document metadata and search
- Integrated third-party APIs and NoSQL storage to support scalable content retrieval
- Improved average query performance by 30% through indexing and caching strategies

"""

sample_jd = """
Job Posting Date
04-08-2026
Job Requisition ID
JR39102
Teams
Engineering
Work Type
Remote

At Netflix, our mission is to entertain the world. Together, we are writing the next episode - pushing the boundaries of storytelling, global fandom and making the unimaginable a reality. We are a dream team obsessed with the uncomfortable excitement of discovering what happens when you merge creativity, intuition and cutting-edge technology. Come be a part of what’s next.

ABOUT YOU
You are an individual who thrives on independence, has a passion for creating high-quality products, and can form close work partnerships with engineers, product managers, and designers. You are adept at problem-solving, enjoy a challenge, and have experience building scalable, high-performing applications. You have excellent communication and collaboration skills, and, most importantly, you have experience with front-end technologies, Node, JavaScript, React. Must have some experience with back-end technologies, SpringBoot, NoSQL, GraphQL, GenAI, LLMs.
In this role, you’ll design and implement intuitive, high-quality front-end experiences while integrating AI and LLM-powered capabilities into our enterprise, employee-facing applications. You’ll be a part of the team chartered with building company-wide tools for the entire workforce and will partner with your peers to execute and deliver on the broader strategy and vision. You'll design, develop, and maintain products, infrastructure, and integrations with third-party and other teams’ services. You will collaborate effectively with myriad teams and communicate technical ideas persuasively. You influence and shape team and project direction through effective partnerships across the entire company to save Netflix time and money and enhance productivity for the whole workforce.
 
RESPONSIBILITIES
Design and implement intuitive UX flows for AI-powered features (e.g., chat interfaces, autocomplete, content generation, smart suggestions).
Build and maintain backend services and APIs (REST/GraphQL/etc.) that power the UI, with attention to scalability and reliability.
Collaborate with cross-functional teams to integrate generative AI solutions into existing workflow systems.
Work cross-functionally to build, test, deploy, and launch UIs that operationalize our workflows at scale.
Collaborate extremely effectively with product managers, designers, other engineers, stakeholders, and vendors on projects within the team and across all of Netflix.
Communicate technical ideas and work closely with other senior members of the team.

WE VALUE
You have strong problem-solving skills and the ability to make data-driven decisions.
You are passionate about engineering principles and developing tools and applications for impact with high quality.
You thrive in a dynamic environment where needs shift and can clearly articulate priorities and critical outcomes to your team.
| SKILLS AND EXPERIENCE
4+ years of experience as a software engineer, demonstrably delivering on time, at quality
Expert knowledge of data structures, algorithms, and modern design patterns and data layers
Knowledge of Python and/or Java
Deep experience with Node, JavaScript, React, SpringBoot, NoSQL, GraphQL
Demonstrable ability to lead a project and deliver an end product on time, at quality
Passion to build internal solutions and own development of enterprise-wide applications
Extensive knowledge of building quality APIs for internal and external products 
Experience integrating internal and third-party services into your solutions
Expert knowledge of cloud computing platforms like Amazon Web Services (AWS), GCP
 
Generally, our compensation structure consists solely of an annual salary; we do not have bonuses. You choose each year how much of your compensation you want in salary versus stock options. To determine your personal top of market compensation, we rely on market indicators and consider your specific job family, background, skills, and experience to determine your compensation in the market range. The range for this role is $225,000.00 - $360,000.00. This compensation range will vary based on location.
Netflix provides comprehensive benefits including Health Plans, Mental Health support, a 401(k) Retirement Plan with employer match, Stock Option Program, Disability Programs, Health Savings and Flexible Spending Accounts, Family-forming benefits, and Life and Serious Injury Benefits. We also offer paid leave of absence programs. Full-time hourly employees accrue 35 days annually for paid time off to be used for vacation, holidays, and sick paid time off. Full-time salaried employees are immediately entitled to flexible time off. See more details about our Benefits here.
Netflix is a unique culture and environment. Learn more here.
Inclusion is a Netflix value and we strive to host a meaningful interview experience for all candidates. If you want an accommodation/adjustment for a disability or any other reason during the hiring process, please send a request to your recruiting partner.
We are an equal-opportunity employer and celebrate diversity, recognizing that diversity builds stronger teams. We approach diversity and inclusion seriously and thoughtfully. We do not discriminate on the basis of race, religion, color, ancestry, national origin, caste, sex, sexual orientation, gender, gender identity or expression, age, disability, medical condition, pregnancy, genetic makeup, marital status, or military service.
Job is open for no less than 7 days and will be removed when the position is filled.
"""

sample_score = compute_ats_score(sample_resume, sample_jd)
sample_score

{'overall_score': 84.25,
 'keyword_overlap_score': 65.0,
 'tech_keyword_score': 100.0,
 'action_verb_score': 100,
 'section_coverage_score': 100.0,
 'important_jd_keywords': ['javascript',
  'node',
  'algorithms',
  'aws',
  'java',
  'python',
  'react',
  'rest',
  'experience',
  'time',
  'compensation',
  'netflix',
  'design',
  'job',
  'knowledge',
  'other',
  'skills',
  'team',
  'teams',
  'work']}

# 7. Evaluation Rubric

## Evaluation Rubric

This section defines the human evaluation rubric used to assess the quality and trustworthiness of model outputs.

Because resume improvement is not purely objective, ATS-alignment score alone is not enough. In addition to the automated score, the project uses human evaluation to judge whether the generated feedback and revisions are actually helpful, accurate, and appropriate.

Each model output will be rated on the following dimensions:

1. **ATS Alignment**  
   How well the revised resume appears to match the target job description in terms of relevant keywords, skills, and role-specific language.

2. **Factual Faithfulness**  
   Whether the model preserves the user’s original facts and avoids inventing experience, metrics, certifications, job titles, dates, or other unsupported claims.

3. **Relevance to Job Description**  
   Whether the feedback or rewrite is clearly aligned with the target software engineering role and focuses on the most important requirements in the job description.

4. **Clarity and Professionalism**  
   Whether the output is clear, well-written, professional, and appropriate for a resume or resume feedback context.

5. **Usefulness of Feedback**  
   Whether the feedback is concrete, actionable, and likely to help the user improve their resume.

### Rating Scale

Each dimension will be rated on a **1 to 5 scale**:

- **1 = Very poor**
- **2 = Poor**
- **3 = Acceptable**
- **4 = Good**
- **5 = Excellent**

### Suggested Interpretation

- A strong output should score well on both automated ATS alignment and human evaluation.
- A response that improves ATS score but introduces false or exaggerated claims should be rated poorly on factual faithfulness.
- Human evaluation is especially important for identifying hallucinations, vague advice, misleading wording, and low-value suggestions.

### Evaluation Procedure

For each test or verification case:

- run the same task on both Qwen and Llama
- review the structured output
- compute the ATS-alignment score before and after revision
- assign human ratings using the rubric above
- record notes about strengths, weaknesses, and possible trustworthiness concerns

This rubric supports the project’s goal of evaluating not only whether the model improves resume-job matching, but also whether the output remains accurate, trustworthy, and useful for human users.

# 8. Single Example Run

### run one case through one or both models and inspect the output.

In [18]:
# Purpose: Run one sample case through the full PolishCV pipeline
def get_revised_resume_text(parsed_output: Dict[str, Any], original_resume_text: str) -> str:
    """
    Build a revised resume text string from parsed model output.

    For now, this uses the rewritten_experience field if available.
    If no rewritten content is present, it falls back to the original resume.
    """
    rewritten_experience = parsed_output.get("rewritten_experience", [])

    if isinstance(rewritten_experience, list) and rewritten_experience:
        return "\n".join(str(item).strip() for item in rewritten_experience if str(item).strip())

    return original_resume_text


def run_single_case(case: Dict[str, Any], model_name: str) -> Dict[str, Any]:
    """
    Run one evaluation case through:
    - input validation
    - prompt generation
    - model inference
    - output parsing
    - ATS score comparison
    """
    case_id = case.get("case_id", "unknown_case")
    task_name = case.get("task_name", DEFAULT_TASK)
    resume_text = clean_text(case.get("resume_text", ""))
    job_description = clean_text(case.get("job_description", ""))

    validation = validate_inputs(resume_text, job_description)
    if not validation["is_valid"]:
        return {
            "success": False,
            "case_id": case_id,
            "model_name": model_name,
            "task_name": task_name,
            "error": f"Input validation failed: {validation['errors']}"
        }

    inference_result = generate_for_task(
        model_name=model_name,
        task_name=task_name,
        resume_text=resume_text,
        job_description=job_description
    )

    if not inference_result["success"]:
        return {
            "success": False,
            "case_id": case_id,
            "model_name": model_name,
            "task_name": task_name,
            "error": inference_result["error"],
            "latency_seconds": inference_result.get("latency_seconds")
        }

    raw_output = inference_result["raw_text"]
    parsed_output = parse_model_output(raw_output)

    revised_resume_text = get_revised_resume_text(parsed_output, resume_text)
    score_comparison = compare_scores(resume_text, revised_resume_text, job_description)

    return {
        "success": True,
        "case_id": case_id,
        "model_name": model_name,
        "task_name": task_name,
        "resume_text": resume_text,
        "job_description": job_description,
        "raw_output": raw_output,
        "parsed_output": parsed_output,
        "revised_resume_text": revised_resume_text,
        "score_comparison": score_comparison,
        "latency_seconds": inference_result.get("latency_seconds"),
        "notes": case.get("notes", "")
    }


def display_single_case_result(result: Dict[str, Any]) -> None:
    """
    Display a single-case result in a readable way.
    """
    if not result["success"]:
        print("Run failed.")
        print("Case ID:", result.get("case_id"))
        print("Model:", result.get("model_name"))
        print("Task:", result.get("task_name"))
        print("Error:", result.get("error"))
        return

    print("=" * 80)
    print("Case ID:", result["case_id"])
    print("Model:", result["model_name"])
    print("Task:", result["task_name"])
    print("Latency (seconds):", result["latency_seconds"])
    print("=" * 80)

    print("\nOriginal Resume:\n")
    print(result["resume_text"])

    print("\nJob Description:\n")
    print(result["job_description"])

    print("\nParsed Model Output:\n")
    print(json.dumps(result["parsed_output"], indent=2))

    print("\nRevised Resume Text:\n")
    print(result["revised_resume_text"])

    print("\nATS Score Comparison:\n")
    print(json.dumps(result["score_comparison"], indent=2))

    if result.get("notes"):
        print("\nCase Notes:\n")
        print(result["notes"])

def parse_model_output(raw_output: str) -> Dict[str, Any]:
    return {
        "summary": "",
        "rewritten_experience": [],
        "missing_keywords": [],
        "feedback": [raw_output],
        "gap_suggestions": [],
        "risk_flags": []
    }

In [19]:
# llama test with sample inputs

sample_resume = """
Computer Science student with experience building Python projects, using Git for version control,
and working with SQL in coursework. Built a small web app for a class project and collaborated
with teammates on debugging and testing.
"""

sample_jd = """
We are hiring an entry-level software engineer with experience in Python, SQL, Git,
debugging, APIs, and teamwork. Candidates should be able to build and improve software systems.
"""

# Build one prompt
prompt = build_resume_feedback_prompt(sample_resume, sample_jd)

# Choose which model to test
llama_result = run_llama(prompt)   # or run_llama(prompt)

print("Success:", llama_result["success"])
print("Model ID:", llama_result["model_id"])
print("Latency:", llama_result["latency_seconds"])

if llama_result["success"]:
    print("\nRAW MODEL OUTPUT:\n")
    print(llama_result["raw_text"])

    parsed = parse_model_output(llama_result["raw_text"])
    print("\nPARSED OUTPUT:\n")
    print(json.dumps(parsed, indent=2))
else:
    print("\nERROR:\n")
    print(llama_result["error"])

Success: False
Model ID: meta-llama/Llama-3.2-1B-Instruct
Latency: None

ERROR:

Client error '401 Unauthorized' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-69ed67c2-17dace444752e73e51b7ec13;bbabc458-d624-4b22-9bcd-dc4fe5268d80)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.


In [ ]:
# # openai test with sample inputs

# # Build one prompt
# prompt = build_resume_feedback_prompt(sample_resume, sample_jd)

# # Choose which model to test
# openai_result = run_openai(prompt)   # or run_openai(prompt)

# print("Success:", openai_result["success"])
# print("Model ID:", openai_result["model_id"])
# print("Latency:", openai_result["latency_seconds"])

# if openai_result["success"]:
#     print("\nRAW MODEL OUTPUT:\n")
#     print(openai_result["raw_text"])

#     parsed = parse_model_output(openai_result["raw_text"])
#     print("\nPARSED OUTPUT:\n")
#     print(json.dumps(parsed, indent=2))
# else:
#     print("\nERROR:\n")
#     print(openai_result["error"])

Success: True
Model ID: gpt-5.4
Latency: None

RAW MODEL OUTPUT:

{
  "summary": "The resume shows a solid baseline match for an entry-level software engineering role because it mentions Python, SQL, Git, debugging, teamwork, and a web app project. However, it is very brief and uses generic wording that may not perform well in ATS or with recruiters. The biggest gap versus the job description is that APIs are not mentioned, and the resume does not clearly describe what was built, improved, or how the candidate contributed.",
  "rewritten

PARSED OUTPUT:

{
  "summary": "",
  "rewritten_experience": [],
  "missing_keywords": [],
  "feedback": [
    "{\n  \"summary\": \"The resume shows a solid baseline match for an entry-level software engineering role because it mentions Python, SQL, Git, debugging, teamwork, and a web app project. However, it is very brief and uses generic wording that may not perform well in ATS or with recruiters. The biggest gap versus the job description is that A